# Cuaderno 5 — Arreglar y crear columnas

**Descripción y Visualización de Datos — UAI 2026 — Clase 5**

La clase pasada terminamos con una mentira. Preguntamos *"¿cuántos se demoran más
de una hora en llegar?"*, R contestó **diez**, y la respuesta estaba mal: dejó
fuera justo a los que más viajan. No hubo ningún error en pantalla.

Hoy arreglamos eso. La herramienta es la misma de siempre, `mutate()`, pero usada
para lo que de verdad ocupa el tiempo de cualquiera que trabaje con datos:

| | Qué hace |
|---|---|
| `mutate()` + `as.numeric()` | **arregla** una columna sucia |
| `mutate()` + `ifelse()` | **clasifica**: convierte un número en una categoría |
| `filter()` con `&` y <code>&#124;</code> | pide **dos condiciones** a la vez |

Nada más. Los verbos de la clase pasada —`filter()`, `select()`, `%>%`— siguen
siendo los mismos; hoy sólo aprenden a hacer dos cosas nuevas.

## Parte 0 — Punto de partida

Nada nuevo: abrir `dplyr` y cargar la base del curso.

In [ ]:
library(dplyr)

curso <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/encuesta_curso.csv")

nrow(curso)

In [ ]:
names(curso)

## Parte 1 — Calentamiento

Tres ejercicios con lo de la clase pasada y nada más. Si alguno no te sale solo,
ése es el que hay que repasar hoy.

**1.** Con pipe: quédate con quienes usan Android y muestra `comuna` y `dominio`.

*(`curso`, luego `filter()` con `sistema_operativo == "Android"`, luego `select()`)*

In [ ]:
# Tu código acá

**2.** Cuenta los dominios de mayor a menor, encadenado con `%>%`.

In [ ]:
# Tu código acá

**3.** Agrega una columna `personas_casa` que sea `hermanos + 1` y muéstrala
al lado de `hermanos`.

In [ ]:
# Tu código acá

## Parte 2 — La columna que mentía

Volvamos al crimen de la clase pasada.

In [ ]:
curso %>%
  filter(minutos_viaje > 60) %>%
  count(minutos_viaje)

Diez personas: 70, 80, 85 y 90 minutos.

¿Y los que escribieron 100, 130, 150 y 180? Son los que **más** se demoran de
todo el curso y el filtro los borró. Para saber por qué, hay una función que
conviene tener siempre a mano: `class()` responde *"¿de qué tipo es esta
columna?"*.

In [ ]:
class(curso$minutos_viaje)

`"character"`. **Texto.** No es un número, aunque se vea como uno.

Y R, cuando compara texto, no mide: **deletrea**. Pone `"130"` al lado de `"60"`,
ve que `1` viene antes que `6` en el abecedario y concluye que 130 es menor que
60. Igual que en una guía de teléfonos.

¿Por qué es texto una columna de minutos? Por una sola respuesta. Míralas todas:

In [ ]:
curso %>% count(minutos_viaje)

Ahí está: **`10 min`**. Una persona de treinta y tres escribió la unidad, y
con eso la columna entera dejó de ser numérica. Fíjate además en el orden en que
salieron los valores —10, 10 min, 100, 130, 15…—: ése es orden de diccionario, no
de números. La tabla te estaba avisando.

El arreglo cabe en una línea. `as.numeric()` convierte texto en número, y va
adentro de un `mutate()` porque lo que queremos es **una columna nueva**:

In [ ]:
curso %>%
  mutate(minutos_num = as.numeric(minutos_viaje)) %>%
  count(minutos_num)

Dos cosas que mirar.

Primero: apareció un **aviso en rojo** —*NAs introduced by coercion*, o "se
introdujeron NA por coerción"—. No es un error, es R avisando que hubo algo que
no supo convertir.

Segundo: en la tabla hay una fila `NA`, con 1 caso. Ése es el `"10 min"`. `NA`
significa **"no sé"**, y es la respuesta honesta: R prefiere decir que no sabe
antes que inventar un número.

> Compara las dos conductas. Como texto, `"10 min"` se comparaba en silencio y
> arrastraba a otros cuatro valores al error. Como `NA`, se declara. **Ese es el
> negocio de limpiar datos: cambiar errores silenciosos por errores visibles.**

Ahora sí, la pregunta original, bien hecha:

In [ ]:
curso %>%
  mutate(minutos_num = as.numeric(minutos_viaje)) %>%
  filter(minutos_num > 60) %>%
  count(minutos_num)

**Catorce**, no diez. Misma pregunta, mismo curso, mismo día. Lo único que
cambió fue una línea de limpieza.

> **La regla del día:** antes de comparar una columna con `>`, `<` o `>=`,
> pregúntale `class()`. Si dice `"character"`, arréglala con `mutate()` antes de
> filtrar.

**4.** `horas_sueno` tiene exactamente el mismo problema. Compruébalo con
`class()`, arréglala con `mutate()` y cuenta cuántas personas duermen **menos de
6 horas**.

In [ ]:
# Tu código acá

**5.** Lo mismo con `horas_redes`: ¿cuántas personas pasan **más de 4 horas
al día** en redes sociales?

In [ ]:
# Tu código acá

## Parte 3 — `ifelse()`: de un número a una categoría

Ya sabes limpiar. Lo otro que hace `mutate()` todo el tiempo es **clasificar**:
tomar una columna y agrupar sus valores en dos etiquetas.

Casi ninguna audiencia quiere saber que alguien viaja 87 minutos. Quiere saber
cuántos tienen un viaje *largo*. Eso se escribe con `ifelse()`:

In [ ]:
curso %>%
  mutate(minutos_num = as.numeric(minutos_viaje),
         viaje = ifelse(minutos_num > 60, "Largo", "Corto")) %>%
  count(viaje)

18 cortos, 14 largos y 1 `NA` (nuestro viejo conocido).

La forma es siempre la misma, y se lee de izquierda a derecha:

```
ifelse( la pregunta , qué poner si es TRUE , qué poner si es FALSE )
```

Dos detalles del código de arriba que vale la pena notar:

- **Un solo `mutate()` puede crear varias columnas**, separadas por coma.
- La segunda usa `minutos_num`, que **acababa de nacer en la línea anterior**.
  `mutate()` trabaja de arriba hacia abajo, así que eso está permitido.

**6.** Crea una columna `dormilon` que diga `"Sí"` para quienes duermen 7 horas o
más y `"No"` para el resto, y cuéntala. *(Acuérdate de limpiar `horas_sueno`
primero.)*

In [ ]:
# Tu código acá

## Parte 4 — Dos condiciones a la vez: `&` y `|`

Hasta ahora cada `filter()` pedía una sola cosa. Se pueden pedir dos:

| Signo | Significa | Se cumple cuando |
|---|---|---|
| `&` | **y** | las dos condiciones son verdaderas |
| <code>&#124;</code> | **o** | basta con que una lo sea |

*"¿Cuántos vienen en micro **y** además se demoran más de una hora?"*

In [ ]:
curso %>%
  mutate(minutos_num = as.numeric(minutos_viaje)) %>%
  filter(transporte == "Micro o bus" & minutos_num > 60) %>%
  nrow()

Diez personas del curso pasan más de una hora arriba de una micro para llegar
a clases. Ése es el tipo de frase que sirve en una plataforma de datos: no es un
promedio, es un grupo concreto y contable.

**7.** Ahora con `|`: ¿cuántas personas usan **metro o micro** (es decir,
transporte público)?

In [ ]:
# Tu código acá

## Parte 5 — La receta completa

Todo junto, respondiendo una pregunta de principio a fin: *"entre quienes usan
transporte público, ¿cuántos tienen viaje largo y qué temas eligieron?"*

In [ ]:
curso %>%
  mutate(minutos_num = as.numeric(minutos_viaje),
         viaje = ifelse(minutos_num > 60, "Largo", "Corto")) %>%
  filter(transporte == "Metro" | transporte == "Micro o bus") %>%
  filter(viaje == "Largo") %>%
  count(dominio, sort = TRUE)

Cinco líneas que se leen como cinco instrucciones en orden: **toma, arregla,
clasifica, recorta, cuenta.** Esa cadena, con otra base y otros nombres de
columna, es el motor del Sprint 2.

**8. Tu propia receta.** Arma una cadena de al menos cuatro pasos que incluya un
`mutate()` de limpieza o de clasificación. Úsala sobre el curso, o mejor todavía:
**sobre la fuente que eligió tu grupo en el Sprint 1.**

In [ ]:
# Tu código acá

### Desafío opcional

`estatura` está peor que `minutos_viaje`: mezcla centímetros (`173`), metros con
punto (`1.58`) y metros con coma (`1,60`). Si conviertes con `as.numeric()` te
van a quedar dos `NA` y varias personas midiendo 1,76 centímetros.

¿Puedes dejar una columna `estatura_cm` donde todos estén en centímetros? Pista:
`ifelse()` sirve para decidir a quién multiplicar por 100.

In [ ]:
# Tu código acá

## Antes de irte

**Archivo → Guardar** (Ctrl+S).

### Lo que aprendiste hoy

| Para qué | Cómo se escribe |
|---|---|
| Saber de qué tipo es una columna | `class(curso$minutos_viaje)` |
| Convertir texto en número | `mutate(minutos_num = as.numeric(minutos_viaje))` |
| Convertir un número en categoría | `mutate(viaje = ifelse(minutos_num > 60, "Largo", "Corto"))` |
| Crear dos columnas de una vez | `mutate(a = ..., b = ...)` |
| Pedir las dos condiciones | `filter(transporte == "Metro" & minutos_num > 60)` |
| Pedir una u otra | <code>filter(dominio == "Salud" &#124; dominio == "Educación")</code> |
| Contar las filas de una cadena | `curso %>% filter(...) %>% nrow()` |

### Las tres ideas

1. **Una respuesta mala arruina la columna entera.** Un `"10 min"` entre treinta
   y tres respuestas convirtió los minutos en texto, y el texto se compara
   deletreando.
2. **`NA` es una buena noticia.** Significa que R encontró algo que no supo leer y
   te lo dijo. El caso peligroso es el que no avisa.
3. **Limpiar no es el paso previo al análisis: es análisis.** Diez versus catorce
   personas es una diferencia del 40%, y salió de una sola línea de `mutate()`.

### Para la próxima clase

`group_by()` y `summarise()`: dejar de contar el curso completo y empezar a
compararlo por grupos —por comuna, por transporte, por año.